# Review Radar — the live demo

One kernel, ten moves, 4:40 of a five-minute budget (ADR-0009, RR-11).

Every cell calls a function in `src/serving/demo.py`; no logic lives here. The moves are the
running order the rehearsal counts, and the cell below prints it — including the moves whose
phase is not built yet, which are **placed and marked**, never dropped.

**Before running anything**, in a real terminal, not in this notebook:

```
make up-ui          # the stack, plus Kibana for move 5
make health         # move 1 — four services proved usable from Python
```

Run this notebook top to bottom. It is committed with outputs stripped; an executed export
under `docs/demo/<date>/` is a rehearsal record (`DEMO_GATE`, ticket 18).

In [1]:
from IPython.display import display

from src.serving import demo

display(demo.run_sheet())
demo.budget()

,move,title,surface,s,says,pending
0,1,The stack is real,terminal,10,"four services proved usable from Python, in a ...",
1,2,The paced replay and the streaming projection,notebook,20,the file goes onto its own topic in event orde...,
2,3,Silver's reconciliation identity,notebook,15,"every bronze row is accounted for: rejected, s...",
3,4,Every run this data went through,notebook,30,the silver row names the bronze snapshot it re...,
4,5,Decline candidates,kibana,45,"the opening question, answered over the servin...",
5,6,Hybrid decomposition,notebook,35,a review neither list ranks first wins the fusion,
6,7,Cached complaint-theme labels for the top decl...,notebook,30,themes are hypotheses about what the text says...,
7,8,The evaluation table,notebook,20,"one row per capability, every bar set before t...",
8,9,"RAG answers, claim by claim",notebook,30,every claim cites a review inside the window t...,
9,10,The stream's own gate,notebook,25,the near slice was accepted and the far slice ...,the stream.alerts table ADR-0010 sketches is n...


{'moves_s': 260,
 'switches_s': 20,
 'live_s': 280,
 'budget_s': 280,
 'ceiling_s': 300,
 'spare_s': 20}

---
## Move 1 — the stack is real · terminal · 10 s

Said out loud: *four services proved usable from Python, in a real shell, before the notebook
opens.*

`make health` runs in the terminal, not here: a `!` cell proves the kernel can shell out, and
what is being claimed is that Kafka, MinIO, Elasticsearch and Postgres answer. The notebook's
own three connections open next, and they stay open for the whole demo.

In [2]:
stage = demo.open_stage(scope="full")
demo.context(stage)

26/09/14 13:18:20 WARN Utils: Your hostname, philips-MacBook-Air-2.local resolves to a loopback address: 127.0.0.1; using 10.60.33.34 instead (on interface en0)
26/09/14 13:18:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/philipbergman/.ivy2/cache
The jars for the packages stored in: /Users/philipbergman/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1c867ded-7ea8-42bb-97f0-29630cc9286b;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.6.1 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka

:: loading settings :: url = jar:file:/Users/philipbergman/Documents/Coding_Projects/Big_Data/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 154ms :: artifacts dl 6ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-aws-bundle;1.6.1 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	org.apache.kafka#kafka-c

,fact,value
0,git_commit,4ccc4e4a933f5eeb7b62d93e87d176dfe66b5df8 (DIRTY)
1,scope,full
2,opened_at,2026-09-14T10:18:21+00:00
3,elasticsearch,http://localhost:9200
4,postgres,localhost:5432/catalog
5,category,All_Beauty


---
## Move 2 — the paced replay and the streaming projection · notebook · 20 s

Said out loud: *the file goes onto its own topic in event order at a constant rate, with
lateness injected in counts frozen before the run; the event-time clock moves while the rest
of the demo runs.*

The replay is a background process (`src/ingest/stream_producer.py --inject`, its own ledger
run) and the projection is this kernel's Spark session running the aggregation job's own
functions. Both keep going through moves 3–9; move 10 reads them.

This is the **demo** run (ADR-0010, ticket 16): it holds the near slice back by 7 days and
the far slice by 730, so the watermark has something to accept and something to drop. It
runs on `reviews.stream.demo`, so `reset=True` never touches the control topic that
`make gate-stream` re-reads — which is also what lets a rehearsal run without rebuilding it.


In [3]:
demo.reset_replay_topic(stage.scope)
live = demo.start_live_projection(stage, replay=demo.start_replay(scope=stage.scope,
                                                                  reset=False))
print("\n".join(demo.replay_clock()))

[stream] RESET: deleting topic 'reviews.stream.demo'


[stream] topic 'reviews.stream.demo' is empty and ready for the replay


[stream] run_kind=demo topic=reviews.stream.demo rate=3,000/s source=All_Beauty.sorted.jsonl sha256=3a532607ee9c… ordered_by=756fb1f2


---
## Move 3 — silver's reconciliation identity · notebook · 15 s

Said out loud: *every bronze row is accounted for: rejected, survived, or removed as a
duplicate.*

These are the lines `make gate-silver` printed, read back from `eval/silver/gate.json`. The
gate is a re-derivation over pinned snapshots and costs minutes; showing its artefact is the
same number without retyping it.

In [4]:
demo.show_gate("silver")

SILVER_REJECT_REASON reason=unparsable_json rows=0
SILVER_REJECT_REASON reason=missing_key_field rows=0
SILVER_REJECT_REASON reason=invalid_rating rows=0
SILVER_REJECT_REASON reason=timestamp_out_of_range rows=0 below_1995=None after_ingest=None
SILVER_COLLISIONS groups=6139 exact_groups=6138 conflicting_groups=1 unresolvable_groups=0 table_rows=13415 removed=7276
SILVER_LEDGER run_id=4447727f-ee66-48eb-be9e-8a23e061d708 snapshots_stamped=true ledger_matches_tables=true commit=3c3162ac66c8869c6192fbc2a38cba29062853c4 dirty=true
SILVER_RERUN previous_run=42a6bd4c-0677-44d4-afdb-2a8765d02767 identical=true
RUN_CONTRACTS gate=SILVER_GATE jobs=silver,catalogue_load registered=2/2 missing=none run_contract_registered=true
SILVER_GATE bronze_snapshot=1672093859820484049 catalogue_load_id=84af7fa0-e6db-4b53-b58e-f4df321e0436 catalogue_rows_read=112590 bronze_rows=701528 reject_rows=0 silver_rows=694252 collision_rows_removed=7276 unmatched_review_rows=0 unmatched_parent_asins=0 join_cardinali

---
## Move 4 — every run this data went through · notebook · 30 s

Said out loud: *the silver row names the bronze snapshot it read, and time travel reads that
snapshot back to the same count.*

Provenance as a join, not an assertion (ADR-0008, RR-16): the ledger row, then a Spark SQL
`VERSION AS OF` read of exactly the snapshot that row pinned.

In [5]:
display(demo.ledger(stage, last=12))
demo.time_travel(stage)

,job,spec,status,started_at,in,out,run_id
51,stream_produce,2,success,2026-09-14 09:00:06.409640+00:00,701528.0,701528.0,b95710c7
52,stream_aggregate,1,success,2026-09-14 09:04:10.624589+00:00,701528.0,404480.0,2d877020
53,stream_produce,2,success,2026-09-14 09:16:33.112143+00:00,701528.0,701528.0,49cd7366
54,stream_aggregate,1,success,2026-09-14 09:20:33.137742+00:00,701528.0,404480.0,fe24b1d5
55,sort_replay,1,success,2026-09-14 09:45:49.221061+00:00,701528.0,701528.0,756fb1f2
56,stream_produce,3,success,2026-09-14 09:46:07.224931+00:00,701528.0,701528.0,b3413602
57,stream_produce,3,success,2026-09-14 09:46:08.210581+00:00,701528.0,701528.0,d8737729
58,stream_aggregate,1,success,2026-09-14 09:50:54.104279+00:00,701528.0,404480.0,74dde108
59,stream_aggregate,1,success,2026-09-14 09:52:02.763135+00:00,701528.0,403648.0,50caffcf
60,stream_produce,3,running,2026-09-14 10:10:32.307022+00:00,NaN,NaN,393bf3a7


,fact,value
0,silver run,4447727f
1,silver records_in,"701,528"
2,lake.bronze.reviews_raw pinned snapshot,1672093859820484049
3,count VERSION AS OF that snapshot,"701,528"
4,matches records_in,true
5,count as the table stands now,"701,528"
6,snapshots on the table,5


---
## Move 5 — decline candidates · Kibana · 45 s

Said out loud: *the opening question, answered over the serving projection.*

The one move that leaves the notebook. The dashboard reads the `product_month` alias and
nothing else; individual reviews are shown from here, never from Kibana.

In [6]:
print(demo.kibana_url())

http://localhost:5601/app/dashboards#/view/pm-dashboard


---
## Move 6 — hybrid decomposition · notebook · 35 s

Said out loud: *a review neither list ranks first wins the fusion.*

One descriptive query from the frozen twenty, run through the production hybrid — the same
function P5 was evaluated on — with the per-leg ranks it already carries unpacked beside the
fused score `Σ 1/(60 + rank)`.

In [7]:
table = demo.hybrid_decomposition(stage, demo.DEMO_QUERY)
display(table)
print(demo.DEMO_QUERY, "→", demo.fusion_story(table))

[live] batch 0: 0 deduplicated rows, event time reached  (0 so far)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,fused_rank,bm25_rank,knn_rank,fused_score,rating,title,review_id
0,1,6.0,3,0.031025,2,Chips too fast,9189dd1d0d04
1,2,24.0,7,0.026830,1,Chipping polish,d48f22f0f546
2,3,25.0,9,0.026257,1,Don’t waste your time . Do a gel,c294977f617f
3,4,30.0,10,0.025397,5,I love this polish,b2d65fdb8e31
4,5,11.0,37,0.024394,1,This nail polish is a rip off.,9399ca6d1074
5,6,29.0,19,0.023894,5,Best polish I have ever used!!,b599ba72d885
6,7,20.0,30,0.023611,2,Nail polish chips off the same day I painted t...,9c853e8bbc80
7,8,16.0,39,0.023259,1,Chipped within 3 days,c2ce370bf88b
8,9,17.0,45,0.022511,1,Disappointed,47bb89c72302
9,10,NaN,1,0.016393,1,Polish chips after 1 week,1f33d01f911c


nail polish chips after one day → the fused winner is ranked 6.0 by BM25 and 3 by kNN: neither list puts it first, and the sum of 1/(60+rank) over both legs does; 9 of the fused top 10 are in both legs


---
## Move 7 — cached complaint-theme labels · notebook · 30 s

Said out loud: *themes are hypotheses about what the text says, never a diagnosed cause, and
P6's published macro-F1 is a reported miss.*

Every row was written by `make label-themes` and is read back from `gold.review_theme_labels`:
the demo makes no model call. The counts are over this product's **labelled** reviews — the
seeded sample frames, not its whole history — so no share, direction or cause is claimed here.
The published P6 number is move 8's.

In [8]:
candidate = demo.top_candidate()
print(demo.candidate_headline(candidate))
demo.theme_evidence(stage, candidate=candidate)

top eligible candidate, decline rank 3: B00RPJZMUM (Xtava Professional Ionic Hair Dryer - Peony Pink Salon Blow ): baseline 2016-10..2017-03, recent 2017-04..2017-09


[live] batch 1: 12,808 deduplicated rows, event time reached 2013-04 (12,808 so far)


,window,theme,labelled_mentions,reviews,quote
0,baseline,(no theme),4,2,
1,baseline,hard_to_use,4,3,clean the venting in the back
2,baseline,not_as_described,3,3,the size is compact - not tiny like the smalle...
3,baseline,unpleasant_scent,3,2,smelling like burning plastic
4,baseline,does_not_work,2,2,only works for a couple of months
5,baseline,overpriced,2,2,To buy a hair dryer
6,baseline,wrong_size_or_fit,2,2,compact - not tiny like the smallest one
7,baseline,irritation_or_harm,1,1,I would hate for my toddler to get ahold of it
8,baseline,poor_build_quality,1,1,blade hitting plastic cover
9,recent,does_not_work,1,1,motor stopped working


---
## Move 8 — the evaluation table · notebook · 20 s

Said out loud: *one row per capability, every bar set before the number beside it existed.*

`make eval-table`, rendered in this kernel. Reproducibility rows block their phase; quality
rows report beside their bars and never block. A capability with no artefact and no written
cut reason makes this table `INCOMPLETE` — silence is not one of the verdicts.

In [9]:
print(demo.evaluation_table())

Reproducibility -- these block their phase
  phase               capability      gate               metric           value  bar  verdict  scope  run_id    note
  P2 Silver           silver          SILVER_GATE        constituents_ok  8      8    PASS     full   4447727f
  P2 Silver           silver_repro    SILVER_REPRO_GATE  constituents_ok  13     13   PASS     full   4447727f
  P3 Gold             gold            GOLD_GATE          constituents_ok  3      3    PASS     full   6e1c0d3a
  P3 Gold             gold_repro      GOLD_REPRO_GATE    constituents_ok  10     10   PASS     full   6e1c0d3a
  P4 Search           search          SEARCH_GATE        constituents_ok  7      7    PASS     full   d2e56c46
  P5 Embeddings       embeddings      EMBED_GATE         constituents_ok  7      7    PASS     full   273e0939
  P6 Themes           themes          THEMES_GATE        constituents_ok  7      7    PASS     full   e39584c3
  P7 RAG              rag             RAG_GATE           consti

---
## Move 9 — RAG answers, claim by claim · notebook · 30 s

Said out loud: *every claim cites a review inside the window the question declared; it
contrasts cited examples and never says a complaint increased.*

The thirty were frozen before any answer existed and answered once. The table below resolves
each citation against that question's own retrieved set and checks the stored `review_month`
against the declared window — the two mechanical rules `RAG_GATE` blocks on, re-run live.

In [10]:
display(demo.rag_questions())
qid = demo.clean_question(family="temporal")
print(qid, "—", demo.question_text(qid))
display(demo.citations(qid))
print(demo.citation_contract(qid))

,question_id,family,answerability,status,outcome,claims
0,product_scoped-01,product_scoped,answerable,succeeded,refused,0
1,temporal-01,temporal,answerable,succeeded,answered,3
2,product_scoped-02,product_scoped,answerable,succeeded,answered,5
3,temporal-02,temporal,answerable,succeeded,answered,4
4,product_scoped-03,product_scoped,answerable,succeeded,answered,4
5,temporal-03,temporal,answerable,succeeded,answered,3
6,product_scoped-04,product_scoped,answerable,succeeded,answered,1
7,temporal-04,temporal,answerable,succeeded,answered,6
8,product_scoped-05,product_scoped,answerable,succeeded,answered,6
9,temporal-05,temporal,answerable,succeeded,answered,3


temporal-02 — What complaints appear in the cited reviews for Godefroy Tint Kit for Spot Coloring, Dark Brown (ASIN B0BM4GX6TT) from 2017-04 to 2017-09, and how do those examples contrast with cited reviews from 2016-10 to 2017-03?


,claim,cite_id,window,review_month,in_declared_window,review
0,1. Review R1 from the baseline window complain...,R1,baseline,2016-11,true,I genuinely wanted to like this product
1,2. Review R6 from the recent window praises th...,R6,recent,2017-07,true,Makes a big difference
2,3. Review R8 from the recent window mentions t...,R8,recent,2017-09,true,Love it.
3,4. Review R9 from the recent window states tha...,R9,recent,2017-08,true,Pretty good tint kit..keeps me from art class ...


RAG_CONTRACT temporal-02: no violation -- every citation resolves and is in scope


---
## Move 10 — the stream's own gate · notebook · 25 s

Said out loud: *the near slice was accepted and the far slice dropped, in exactly the counts
frozen before the run, and every differing product-month is explained by a dropped row.*

The live query started at move 2 is stopped and says what it saw; then the recorded
`STREAM_GATE` for the demo run — re-derived by `make gate-stream-demo` from the topic, the
frozen slice counts in `conf/stream_replay.toml`, the sort job's ledger row and the pinned
gold snapshot, and passing only with the control run present beside it.

**Placed and marked:** the `stream.alerts` table ADR-0010 sketches is not built. This move
shows the gate and the reconciliation it carries, not an alerts feed.


In [11]:
display(demo.stop_live_projection(live, stage=stage))
demo.show_gate("stream_demo")

[live] batch 2: 14,865 deduplicated rows, event time reached 2014-09 (27,673 so far)


[live] batch 3: 14,844 deduplicated rows, event time reached 2015-03 (42,517 so far)


[live] batch 4: 14,833 deduplicated rows, event time reached 2015-09 (57,350 so far)


[live] batch 5: 14,837 deduplicated rows, event time reached 2016-01 (72,187 so far)


[live] batch 6: 14,839 deduplicated rows, event time reached 2016-04 (87,026 so far)


...

[live] batch 47: 13,116 deduplicated rows, event time reached 2023-09 (692,252 so far)


[live] batch 48: 0 deduplicated rows, event time reached  (692,252 so far)


DEMO_LIVE topic=reviews.stream.demo batches=49 rows=692252 product_months=0 event_time_reached=none elapsed_s=254


,fact,value
0,topic,reviews.stream.demo
1,micro-batches,49
2,deduplicated rows,"692,252"
3,event time reached,
4,elapsed_s,254


STREAM_SOURCE run_kind=demo replay_run=d8737729 topic=reviews.stream.demo records_acked=701528 sorted_file_sha256=3a532607ee9c replay_config_hash=4c8bccff40bd
STREAM_VALIDATION function=src.spark.silver.parse_and_validate digest=3940f34067d6 silver_digest=3940f34067d6 shared=true
STREAM_DEDUPE records_read=701528 ledger_records_read=701528 rows_valid=701528 rows_rejected=0 unique_review_ids=692252 duplicates_dropped=7276 expected_from_sort=7276 sort_run=756fb1f2
STREAM_WATERMARK delay=30 days injected_drops=2000 source_order=event_time_sorted ingested_at=2026-09-14T09:46:08.210581+00:00
STREAM_SURVIVOR silver_run=4447727f collision_groups=6139 exact=6138 conflicting=1 unresolvable=0 stream_rule=first_arrival batch_rule=helpful_vote,text_length,canonical_hash
STREAM_RECONCILE gold_run=6e1c0d3a product_months_compared=193630 differing=1013 only_in_stream=0 only_in_gold=309 only_in_stream_unmaterialised=210018 columns=review_count,rating_sum,neg_count,verified_count,verified_rating_sum,no

---
## After the clock stops

The exactly-once proof is not a live move (65 s of 300). It is a figure in the design doc and
a terminal transcript in the recorded backup, played only if asked:
`make eos`. A live `llama3.2:3b` label is a Q&A reserve, never on the clock.

In [12]:
demo.close_stage(stage)